In [12]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/sms-spam-collection-dataset


In [7]:
import pandas as pd
import nltk
import numpy as np

In [8]:
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_r

True

In [9]:
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk import sent_tokenize

In [10]:
df = pd.read_csv('/kaggle/input/sms-spam-collection-dataset/spam.csv', encoding='latin-1')

In [11]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [12]:
df.describe()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
count,5572,5572,50,12,6
unique,2,5169,43,10,5
top,ham,"Sorry, I'll call later","bt not his girlfrnd... G o o d n i g h t . . .@""","MK17 92H. 450Ppw 16""","GNT:-)"""
freq,4825,30,3,2,2


In [13]:
df = df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])
display(df.head())

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [14]:
df['v2']

,v2
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."
...,...
5567,This is the 2nd time we have tried 2 contact u...
5568,Will Ì_ b going to esplanade fr home?
5569,"Pity, * was in mood for that. So...any other s..."
5570,The guy did some bitching but I acted like i'd...


In [15]:
message_loc = df.loc[2, 'v2']
print("Message using .loc:", message_loc)

Message using .loc: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's


In [16]:
df['v2'] = df['v2'].str.lower()
df['v2']

,v2
0,"go until jurong point, crazy.. available only ..."
1,ok lar... joking wif u oni...
2,free entry in 2 a wkly comp to win fa cup fina...
3,u dun say so early hor... u c already then say...
4,"nah i don't think he goes to usf, he lives aro..."
...,...
5567,this is the 2nd time we have tried 2 contact u...
5568,will ì_ b going to esplanade fr home?
5569,"pity, * was in mood for that. so...any other s..."
5570,the guy did some bitching but i acted like i'd...


In [17]:
stop_words = stopwords.words('english')
stop_words


['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [18]:
def apply_stopwords(text):
    return [word for word in text if word not in stop_words]

In [19]:
def apply_tokenizer(text):
    return word_tokenize(text)

In [20]:
df['v2'] = df['v2'].apply(apply_tokenizer)
df['v2'] = df['v2'].apply(apply_stopwords)
df['v2']

,v2
0,"[go, jurong, point, ,, crazy, .., available, b..."
1,"[ok, lar, ..., joking, wif, u, oni, ...]"
2,"[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,"[u, dun, say, early, hor, ..., u, c, already, ..."
4,"[nah, n't, think, goes, usf, ,, lives, around,..."
...,...
5567,"[2nd, time, tried, 2, contact, u., u, å£750, p..."
5568,"[ì_, b, going, esplanade, fr, home, ?]"
5569,"[pity, ,, *, mood, ., ..., suggestions, ?]"
5570,"[guy, bitching, acted, like, 'd, interested, b..."


In [21]:
x = df.loc[2,'v2']
type(x[3])

str

In [28]:
pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 10.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tsfresh 0.21.0 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
thinc 8.3.6 req

In [1]:
import gensim.downloader as api

model = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [3]:
model['entry']

array([-0.12792969, -0.07714844, -0.26367188, -0.08935547,  0.33398438,
        0.22851562, -0.15039062, -0.15234375,  0.01928711, -0.11523438,
        0.00305176, -0.18554688,  0.06640625, -0.4296875 , -0.15820312,
       -0.18652344,  0.09619141,  0.00909424,  0.06835938, -0.16796875,
        0.21972656, -0.00836182, -0.23730469, -0.22363281, -0.15820312,
       -0.16015625, -0.00765991,  0.00135803,  0.09082031, -0.05395508,
        0.04125977, -0.35351562,  0.10742188, -0.1640625 ,  0.23535156,
       -0.23144531, -0.03466797, -0.05664062, -0.07421875, -0.17382812,
       -0.06396484,  0.01538086,  0.39648438,  0.23828125,  0.16601562,
       -0.14160156, -0.25976562,  0.40039062, -0.23242188,  0.09472656,
       -0.09619141, -0.05737305, -0.14941406, -0.06030273,  0.13183594,
       -0.06542969,  0.02978516,  0.0062561 , -0.02185059,  0.07714844,
        0.01745605, -0.10107422, -0.10644531, -0.18945312, -0.20117188,
        0.02331543,  0.05932617,  0.3203125 , -0.02258301,  0.11

In [4]:
all_words = model.index_to_key
print(all_words[:100]) # print the first 10 words

['</s>', 'in', 'for', 'that', 'is', 'on', '##', 'The', 'with', 'said', 'was', 'the', 'at', 'not', 'as', 'it', 'be', 'from', 'by', 'are', 'I', 'have', 'he', 'will', 'has', '####', 'his', 'an', 'this', 'or', 'their', 'who', 'they', 'but', '$', 'had', 'year', 'were', 'we', 'more', '###', 'up', 'been', 'you', 'its', 'one', 'about', 'would', 'which', 'out', 'can', 'It', 'all', 'also', 'two', 'after', 'first', 'He', 'do', 'time', 'than', 'when', 'We', 'over', 'last', 'new', 'other', 'her', 'people', 'into', 'In', 'our', 'there', 'A', 'she', 'could', 'just', 'years', 'some', 'U.S.', 'three', 'million', 'them', 'what', 'But', 'so', 'no', 'like', 'if', 'only', 'percent', 'get', 'did', 'him', 'game', 'back', 'because', 'now', '#.#', 'before']


In [5]:
def avg_word2vec(arr, model):
    valid_vectors = [model[word] for word in arr if word in model]

    if not valid_vectors:
        return np.zeros(model.vector_size)

    return np.mean(valid_vectors, axis=0)

In [22]:
df['v2'] = df['v2'].apply(lambda x: avg_word2vec(x, model))
df.loc[3,'v2'].shape

(300,)

In [23]:
df.head()

,v1,v2
0,ham,"[-0.019805908, 0.05167062, 0.02709961, 0.21868..."
1,ham,"[-0.06323496, 0.0803833, 0.060943604, 0.102498..."
2,spam,"[-0.03482437, -0.00703014, -0.06348601, 0.1161..."
3,ham,"[-0.06568061, 0.0262146, 0.1081543, 0.0869751,..."
4,ham,"[0.01461792, 0.07184219, -0.005203247, 0.14686..."


In [25]:
df

,v1,v2
0,ham,"[-0.019805908, 0.05167062, 0.02709961, 0.21868..."
1,ham,"[-0.06323496, 0.0803833, 0.060943604, 0.102498..."
2,spam,"[-0.03482437, -0.00703014, -0.06348601, 0.1161..."
3,ham,"[-0.06568061, 0.0262146, 0.1081543, 0.0869751,..."
4,ham,"[0.01461792, 0.07184219, -0.005203247, 0.14686..."
...,...,...
5567,spam,"[0.044520788, 0.02306257, 0.027987888, 0.05183..."
5568,ham,"[0.021606445, 0.090625, 0.109191895, 0.0634216..."
5569,ham,"[0.03214264, 0.112976074, 0.002380371, 0.07507..."
5570,ham,"[0.09937395, 0.029209683, -0.007853917, 0.0980..."


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [27]:
le = LabelEncoder()
df['v1'] = le.fit_transform(df['v1'])
df
#ham is encoded as 0 and spam is encoded as 1

,v1,v2
0,0,"[-0.019805908, 0.05167062, 0.02709961, 0.21868..."
1,0,"[-0.06323496, 0.0803833, 0.060943604, 0.102498..."
2,1,"[-0.03482437, -0.00703014, -0.06348601, 0.1161..."
3,0,"[-0.06568061, 0.0262146, 0.1081543, 0.0869751,..."
4,0,"[0.01461792, 0.07184219, -0.005203247, 0.14686..."
...,...,...
5567,1,"[0.044520788, 0.02306257, 0.027987888, 0.05183..."
5568,0,"[0.021606445, 0.090625, 0.109191895, 0.0634216..."
5569,0,"[0.03214264, 0.112976074, 0.002380371, 0.07507..."
5570,0,"[0.09937395, 0.029209683, -0.007853917, 0.0980..."


In [28]:
x = df['v2']
y = df['v1']
y

,v1
0,0
1,0
2,1
3,0
4,0
...,...
5567,1
5568,0
5569,0
5570,0


In [29]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [30]:
x_train.shape

(4457,)

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [32]:
log = LogisticRegression(verbose = 3)
log.fit(x_train.tolist(), y_train)

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.2s finished


LogisticRegression(verbose=3)

In [33]:
y_pred = log.predict(x_test.tolist())
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9452914798206278


In [34]:
def predict_message_class(model, w2vmodel, message):
    message = message.lower()
    message = word_tokenize(message)
    message = [word for word in message if word not in stop_words]
    message_vector = avg_word2vec(message, w2vmodel)
    prediction = model.predict([message_vector])
    return prediction[0]


In [35]:
ham_sentence = ["Hey, are you free for lunch tomorrow?",
                "Could you please send me the report by end of day?",
                "Just letting you know I'll be home late tonight.",
                "Did you remember to pick up the groceries?",
                "Let's catch up soon, it's been a while!"]

spam_sentence = ["WINNER! You have won a FREE iPhone! Click here to claim now:",
                "Urgent: Your account has been compromised. Verify your details immediately.",
                "Claim your exclusive discount! Limited time offer, don't miss out!",
                "Congratulations! You've been selected for a prize. Reply STOP to opt out.",
                "Get rich quick! Invest in our amazing new opportunity!"]

In [36]:
print(predict_message_class(log, model, ham_sentence[3]))
print(predict_message_class(log, model, spam_sentence[2]))

0
1
